# MLSM Lab 1 — Entrenamiento interactivo

El notebook se encarga de la interacción y la presentación. La lógica está en `run_lab1.py` y las definiciones de modelos en `model_definitions.py`. Ejecuta las celdas en orden.

In [ ]:
from run_lab1 import (
    load_dataset, select_models, evaluate_models,
    get_confusion_data, create_cv, display_results, plot_confusion_matrix,
    save_run_outputs,
)

## 1. Seleccionar dataset

In [ ]:
print("1 -> GenreAll | 2 -> features-30-sec | 3 -> features-3-sec")
dataset_choice = input("Dataset [1/2/3]: ").strip()

## 2. Cargar datos

In [ ]:
dataset_name, filename, X, y, class_counts = load_dataset(dataset_choice)
print(f"{dataset_name}: {len(y)} muestras, {X.shape[1]} características")
display_results(class_counts.to_frame())

## 3. Seleccionar modelo

In [ ]:
print("l -> Logistic Regression | k -> k-NN | d -> Decision Tree | a -> ANN | all -> Todos")
model_choice = input("Modelo [l/k/d/a/all]: ").strip()
selected_models = select_models(dataset_name, X.shape[1], model_choice)
print("Modelos:", ", ".join(selected_models))

## 4. Entrenar y evaluar

Se utiliza validación cruzada estratificada de 10 particiones. Esta celda puede tardar varios minutos.

In [ ]:
cv = create_cv()
results_df = evaluate_models(X, y, selected_models, cv)
print(f"Configuraciones evaluadas: {len(results_df)}")

## 5. Resultados

In [ ]:
display_results(results_df)

## 6. Seleccionar fila para la matriz de confusión

La matriz usa predicciones *out-of-fold* de validación cruzada.

In [ ]:
row_choice = input(f"Índice de fila [0–{len(results_df)-1}] o q para omitir: ").strip().lower()

## 7. Matriz de confusión

In [ ]:
fig = None
if row_choice == "q":
    print("Matriz omitida.")
else:
    row_index = int(row_choice)
    matrix, classes, row = get_confusion_data(X, y, selected_models, results_df, row_index, cv)
    print(f"{row['Algorithm']} | CV accuracy: {row['CV accuracy']:.4f} | Overfitting gap: {row['Overfitting gap']:.4f}\nParámetros: {row['Parameters']}")
    fig = plot_confusion_matrix(matrix, classes)

## 8. Guardar resultados (opcional)

Esta celda es independiente y se puede volver a ejecutar después de terminar el análisis. Si eliges guardar, se creará un Excel con las tablas de resultados y configuración, y la figura de la matriz de confusión dentro de `Lab1/results/`.

In [ ]:
save_run_outputs(results_df, dataset_name, selected_models, fig)